## Data Exploration Notebook for IFood Data Science Technical Case

## 1. Import Libraries

In [0]:
import numpy as np
from itertools import chain
from collections import Counter
import matplotlib.pyplot as plt

from pyspark.sql.functions import col, sum, when, count, to_date, year, avg, from_json, coalesce, corr

## 2. Utilities

In [0]:
def get_columns_with_nulls(dataframe):
    return dataframe.select([sum(col(c).isNull().cast("int")).alias(c) for c in dataframe.columns])

def info_alternative(dataframe):
    n_rows = dataframe.count()
    print(f"Number of rows: {n_rows}")
    print(f"Number of columns: {len(dataframe.columns)}")
    number_of_nans_in_cols = get_columns_with_nulls(dataframe)
    print("#\tColumn\t\tNon-Null Count\t\tDtype")
    print("---\t------\t\t--------------\t\t-----")
    for index, (col, col_type) in enumerate(dataframe.dtypes):
        print(f"{index}\t{col}\t\t{n_rows - number_of_nans_in_cols.first()[col]}\t\t{col_type}")


In [0]:
def plot_bars(axes, data_values, data_labels, data_colors, title, x_label, y_label):
    axes.bar(x=data_labels, height=data_values, color=data_colors)
    axes.set_title(title)
    axes.set_xlabel(x_label)
    axes.set_ylabel(y_label)

    return axes

def plot_hist(axes, data_values, n_bins, title, x_label, y_label, data_mean=None, data_median=None, mean_color="tab:red", median_color="tab:green"):

    axes.hist(data_values, bins=n_bins)

    if data_mean is not None:
        axes.axvline(x=data_mean, color=mean_color, linestyle="dashed")
        axes.text(data_mean, axes.get_ylim()[1]-100, "Média", color=mean_color)
    if data_median is not None:
        axes.axvline(x=data_median, color=median_color, linestyle="dashed")
        axes.text(data_median, axes.get_ylim()[1]-200, "Mediana", color=median_color)

    axes.set_title(title)
    axes.set_xlabel(x_label)
    axes.set_ylabel(y_label)

    return axes

## 3. Load Data

In [0]:
path_offers = "/Volumes/workspace/ifood/ifood/offers.json"
path_profile = "/Volumes/workspace/ifood/ifood/profile.json"
path_transactions = "/Volumes/workspace/ifood/ifood/transactions.json"

df_offers = spark.read.json(path_offers)
df_profile = spark.read.json(path_profile)
df_transactions = spark.read.json(path_transactions)

print(f"There is {df_offers.count()} offers, {df_profile.count()} profiles and {df_transactions.count()} transactions")

## 4. Exploring the Data

A primeira coisa que faremos é analisar uma amostra dos dados. Isso nos permitirá ter uma compreensão inicial de seus campos e conteúdos. Além disso, às vezes também é possível começar a notar alguns tratamentos necessários, como nos **Dados de Perfil**, que provavelmente contêm algumas idades incorretas registradas (por exemplo, 118 anos) e valores NaN no **limite do cartão de crédito**.

### 4.1. Offers Table

In [0]:
display(df_offers)

Podemos observar que um mesmo tipo de oferta pode ser realizada em diferentes configurações, a depender do tipo de canal de comunicação, valor mínimo para que ela possa ser usada, duração que ela passa disponível e valor do desconto.

Os três tipos de ofertas disponíveis são:
- **Bogo**: que são ofertas do tipo compre um e leve 2.
- **Discount**: que são descontos sobre produtos.
- **Informational**: são anúncios sobre produtos existentes. Logo, não possuem valor mínimo nem valor de desconto.

Os tipos de ofertas que mais aparecem na base são **Bogo** e **Discount**, e a primeira é a que oferece os maiores valores médios de desconto.

In [0]:
figure, (ax1, ax2) = plt.subplots(1,2, figsize=(8,3))
bar_colors = ['tab:green', 'tab:blue', 'tab:orange']
bar_values1 = df_offers.groupBy("offer_type").agg({"discount_value": "avg"}).collect()
bar_labels = [row["offer_type"] for row in bar_values1]
bar_values1 = [row["avg(discount_value)"] for row in bar_values1]

bar_values2 = df_offers.groupBy("offer_type").agg({"id": "count"}).select("count(id)").collect()
bar_values2 = [row["count(id)"] for row in bar_values2]

plot_bars(axes=ax1, data_values=bar_values1, data_labels=bar_labels, data_colors=bar_colors, title="Valor do Desconto x Tipo de Oferta", x_label="Tipo de Oferta", y_label="Valor do Desconto (R$)")

plot_bars(axes=ax2, data_values=bar_values2, data_labels=bar_labels, data_colors=bar_colors, title="Quantidade de Ofertas x Tipo de Oferta", x_label="Tipo de Oferta", y_label="Quantidade de Ofertas")

plt.show()

Em relação ao formato dos dado, vemos que todas as colunas estão com os tipos de dados esperados.

In [0]:
info_alternative(df_offers)

Vemos também que:
- Uma oferta pode durar de 3 a 10 dias, com uma duração média de 6 dias.
- O valor mínimo para uma oferta poder ser aplicada, quando ela possui, costuam ser em torno de $R\$ 8,00$. Mas pode chegar até a $R\$ 20,00$.
- Os descontos, quando existentes, costumam variar de $R\$ 2,00$ a $R\$ 10,00$.

In [0]:
df_offers.describe().show()

### 4.2. Profiles Table

In [0]:
display(df_profile)

Com relação à tabela de informações dos usuários (**Profile**), vemos que ela possui valores faltantes para **gênero** e limite do **cartão de crédito**. 

No caso do gênero, esses valores estão marcados como **None**. Já no caso do cartão de crédito, eles estão como **NaN**. Talvez o usuário tenha decidido não informar, se não fosse obrigatório.

Com relação aos tipos dos dados, quase todos estão corretos, com exceção do campo **registered_on**, que apesar de originalmente estar como um número inteiro, poderia ser melhor representado como um **datetime**, já que indica a data em que o usuário se registrou na plataforma. Além disso, percebemos que ele está seguindo o formato YYYYMMDD.

In [0]:
info_alternative(df_profile)

Com relação ao gênero, temos 3 categorias, além das amostras marcadas como nulas:
- M: masculino
- F: feminino
- O: outros

Alguns clientes não informaram o gênero (ou houve algum problema na coleta) e eles estão com valores nulos. Como esses casos representam quase 13% da base (2175/17000), resolvemos subsituir os valores nulos pela sigla **"NI" (Não Informado)**, ao invés de deletá-los da base.

O mesmo se aplica para o valor do **limite do cartão de crédito**. Mas nestes casos, subsituimos os valores nulos por zero.

Apesar de termos alguma diversidade de gêneros na base de dados, **a maioria dos clientes registrados na plataforma no período de coleta dos dados são do sexo masculino**, seguidos de pessoas do sexo feminino, não informado (NI) e outros.

Com relação ao poder aquisitivo, apesar de não serem maioria na base, clientes do sexo feminino são as que demonstrarm maior valor médio de limite não cartão de crédito, seguidas de pessoas que se identificam com outros gêneros e pessoas do sexo masculino.

Pessoas que não indicaram o gênero também não indicaram o valor do limite do cartão de crédito. Isso pode indicar que, se estas opções forem opcionais no momento do cadastro, essas pessoas resolveram pular, preenchendo apenas o básico.

In [0]:
df_profile = df_profile.withColumn(
    "gender",
    when(col("gender").isNull(), "NI").otherwise(col("gender"))
)
gender_counts = (
    df_profile.groupBy("gender")
    .agg(count("*").alias("count"))
)

df_profile = df_profile.withColumn(
    "credit_card_limit",
    when(col("credit_card_limit").isNull(), 0).otherwise(col("credit_card_limit"))
)

In [0]:
figure, (ax1, ax2) = plt.subplots(1,2, figsize=(12,3))
bar_colors = ['tab:green', 'tab:blue', 'tab:red', 'tab:orange']

gender_counts = gender_counts.collect()
bar_values1 = [row["count"] for row in gender_counts]
bar_labels = [row["gender"] for row in gender_counts]

bar_values2 = df_profile.groupBy("gender").agg({"credit_card_limit": "avg"}).collect()
bar_values2 = [row["avg(credit_card_limit)"] for row in bar_values2]

plot_bars(axes=ax1, data_values=bar_values1, data_labels=bar_labels, data_colors=bar_colors, title="Quantidade de Clientes x Gênero", x_label="Gênero", y_label="Quantidade de Clientes")

plot_bars(axes=ax2, data_values=bar_values2, data_labels=bar_labels, data_colors=bar_colors, title="Valor Médio do Cartão de Crédito x Gênero", x_label="Gênero", y_label="Valor Médio do Limite (R$)")

Com relação à data em que as pessoas se registraram na plataforma, ao invés de subsituir a coluna original, criamos uma nova coluna com os valores convertidos para o tipo **datetime**, o que facilita alguns tipos de manipulações. 

Além disso, podemos perceber, conforme o gráfico abaixo, que houve um crescimento de clientes registrados na plataforma ao longo dos anos, o que é um bom sinal de aceitação da plataforma. Apesar disso, o gráfico também mostra uma queda na quantidade de clientes que se registraram em 2018, o que talvez seja explicado pelo fato de os dados terem sido coletados apenas até julho. Então, é possível que esse valor mude ao coletarmos os dados do restante do ano.



In [0]:
df_profile = df_profile.withColumn("registered_on_dt", to_date("registered_on", "yyyyMMdd"))

df_profile.describe().show()

In [0]:
years = (
    df_profile
    .withColumn("year", year(col("registered_on_dt")))
    .groupBy("year")
    .count()
    .orderBy("year")
).toPandas()

fig, ax = plt.subplots(1,1)
plot_bars(axes=ax, data_values=years["count"], data_labels=years["year"], data_colors=['tab:green'], title="Quantidade de Clientes Registrados ao Longo dos Anos", x_label="Ano", y_label="Quantidade de Clientes Registrados")


Também podemos perceber que um número considerável de clientes está com idade maior que 100 anos, ao passo que 75% dos clientes possuem apenas até 73 anos. Isso acaba puxando a média das idades um pouco mais para cima, conforme podemos ver nos gráficos abaixo.

Esses valores possivelmente resultam de algum problema na coleta dos dados ou entrada do usuário, já que o número de pessoas que vivem mais de 100 anos é pequeno em relação ao total da população, mas eles estão representando 13% dos nossos dados. Tendo isso em vista, resolvemos subsituir os valores maiores que 100 anos pela mediana das idades, para tornar os dados um pouco mais coerentes.

In [0]:
ages = df_profile.select("age").toPandas()["age"]
age_mean = df_profile.select(avg("age")).first()[0]
age_median = df_profile.approxQuantile("age", [0.5], 0.01)[0]

fig, ax = plt.subplots(1,1)
plot_hist(axes=ax, data_values=ages, n_bins=100, title="Distribuição de Idades", x_label="Idade", y_label="Quantidade de Clientes", data_mean=age_mean, data_median=age_median)

plt.show()

In [0]:
df_profile = df_profile.withColumn(
    "age",
    when(col("age") > 100, age_median).otherwise(col("age"))
)

df_profile.describe().show()

In [0]:
ages = df_profile.select("age").toPandas()["age"]
age_mean = df_profile.select(avg("age")).first()[0]
age_median = df_profile.approxQuantile("age", [0.5], 0.01)[0]

fig, ax = plt.subplots(1,1)
plot_hist(axes=ax, data_values=ages, n_bins=10, title="Distribuição de Idades", x_label="Idade", y_label="Quantidade de Clientes", data_mean=age_mean, data_median=age_median)

plt.show()

### 4.3. Transactions Table

Em relação aos **Dados de Transações**, podemos observar que não há campos com valores NaN e que os tipos de dados estão corretos. 

No entanto, no campo **value**, que possui dicionários como valores, estes apresentam alguns problemas, como chaves muito semelhantes com valores diferentes (por exemplo, "offer id" e "offer_id") e valores NaN em alguns campos (por exemplo, em "amount").

In [0]:
display(df_transactions)

In [0]:
info_alternative(df_transactions)

Com o objetivo de facilitar a manipulação dos dados de transação, extraímos os dicionários do campo **value** e inserimos seus valores como novas colunas do dataframe de transações.

In [0]:
df_transactions = df_transactions.select("*", col("value.*")).drop("value")

display(df_transactions)

Como podemos ver, **"offer id"** e **"offer_id"** parecem ter a mesma finalidade, mas são escritos de forma ligeiramente diferente e têm valores diferentes para a maioria de suas entradas. Além disso, quando um deles é NaN, o outro tem valores não Nan.

In [0]:
not_nan_offers_underline = df_transactions.filter(col("offer_id").isNotNull()).select("offer id", "offer_id")
not_nan_offers_space = df_transactions.filter(col("offer id").isNotNull()).select("offer id", "offer_id")

print(f"O campo 'offer id' possui {not_nan_offers_space.count()} valores não nulos e o campo 'offer_id' possui {not_nan_offers_underline.count()} valores não nulos.")

In [0]:
info_alternative(not_nan_offers_underline)

In [0]:
info_alternative(not_nan_offers_space)

Pensando nisso, resolvemos uni-los em um único campo, chamado **offer_id**, preenchendo todos os valores NaN de **offer_id** com os valores não nulos de **offer id** na posição respectiva. Além disso, removemos as colunas **offer id**, pois não é mais necessária.

In [0]:
df_transactions = df_transactions.withColumn("offer_id", coalesce(col("offer_id"), col("offer id"))).drop("offer id")
df_transactions.show(5)

In [0]:
info_alternative(df_transactions)

Com relação aos tipos de eventos registrados nos dados de **Transações**, temos 4 tipos:
- **transaction**: diz respeito a transações realizadas pelo usuário na plataforma, como a compra de um produto.
- **offer received**: refere-se a quando um usuário recebe uma oferta.
- **offer viewed**: refere-se a quando o usuário viu a oferta recebida.
- **offer completed**: refere-se a quando o usuário utiliza a oferta vista.

De acordo com o gráfico de distribuição da quantidade de eventos por tipo, vemos que **compras (transaction)** são o tipo mais comum registrado na base. 

Além disso, o número de ofertas decresce conforme avançamos nas etapas de **receber**, **ver** e **usar/completar**, com apenas **75,7% das ofertas recebidas sendo vistas e apenas 58,2% das ofertas vistas sendo utilizadas**.

In [0]:
event_counts = (
    df_transactions.groupBy("event")
    .agg(count("*").alias("count"))
)

event_counts = event_counts.collect()
bar_values = [row["count"] for row in event_counts]
bar_labels = [row["event"] for row in event_counts]
bar_colors = ["tab:green", "tab:blue", "tab:orange", "tab:red"]

fig, ax = plt.subplots(1,1)
plot_bars(axes=ax, data_values=bar_values, data_labels=bar_labels, title="Quantidade de Eventos x Tipo de Evento", x_label="Tipo de Evento", y_label="Quantidade de Eventos", data_colors=bar_colors)

plt.show()

In [0]:
events = [row["event"] for row in event_counts]
counts = [row["count"] for row in event_counts]
index_received, index_viewed, index_completed = events.index("offer received"), events.index("offer viewed"), events.index("offer completed")

print(f"Porcentagem das ofertas recebidas que são vistas: {100*(counts[index_viewed]/counts[index_received])}%")
print(f"Porcentagem das ofertas vistas que são completadas: {100*(counts[index_completed]/counts[index_viewed])}%")

Na base de dados, temos valores das ofertas apenas para aquelas que foram completadas, mas temos valores para todos os eventos do tipo transaction.

Também, conforme podemos observar nos gráficos abaixo, as ofertas com recompensas de até 5 reais são as que mais frequentemente os usuários utilizam. Um possível fator poderia ser as ofertas de 10 reais exigirem um valor mínimo maior, porém, conforme vemos na correlação calculada (~0.5), as variáveis **min_value** e **discount_value** possuem uma correlação moderada, significando que nem sempre esse aumento é o caso. 

Além disso, a maior parte das compras na plataforma possuem valores até 10 reais, com algumas poucas passando de 30 reais, o que indica que a maioria dos clientes escolhem os produtos mais baratos.

In [0]:
df_transactions.describe().show()

In [0]:
event_offers_completed = df_transactions.filter(col("event") == "offer completed")
event_transactions = df_transactions.filter((col("event") == "transaction") & (col("amount") < 50))

rewards = [row["reward"] for row in event_offers_completed.select("reward").collect()]

amounts = [row["amount"] for row in event_transactions.select("amount").collect()]
amounts_mean = np.mean(amounts)
amounts_median = np.median(amounts)


figure, (ax1, ax2) = plt.subplots(1,2, figsize=(12,3))

plot_hist(axes=ax1, data_values=rewards, title="Quantidade de Ofertas Completadas x Valor da Oferta", x_label="Valor da Oferta", y_label="Quantidade de Ofertas Completada", n_bins=10)
plot_hist(axes=ax2, data_values=amounts, title="Quantidade de Transações (Compras) x Valor da Transação", x_label="Valor da Transação", y_label="Quantidade de Transações", data_mean=amounts_mean, data_median=amounts_median, n_bins=50)

plt.show()

In [0]:
corr_min_value_discount = df_offers.select(corr("min_value", "discount_value").alias("correlation")).first()["correlation"]
print("Correlação entre valor mínimo e valor do desconto:", corr_min_value_discount)